# Entrega 2 — Parte A: Formulación del modelo de *lot-sizing* capacitado

**Caso integrador Bebidas del Maipo S.A.** · Gestión de Producción de Bienes y Servicios · USACH · Grupo 2

Este notebook deduce, paso a paso, el modelo de planificación de producción e inventario para las
13 semanas del horizonte táctico (28/09/2026 a 21/12/2026). Cada uno de los ocho elementos mínimos
del enunciado (sección 7.1) tiene su propia sección: primero se explica la idea, luego se deduce la
restricción y al final se muestra con los datos.

| Punto | Elemento del enunciado | Bloque en `entrega2/modelo.py` |
| :--- | :--- | :--- |
| 1 | Balance de inventario con *backlogging* y rendimiento | `r1_balance` |
| 2 | Variables binarias de preparación y cota $M_{it}$ justificada | `r2_preparacion` |
| 3 | Capacidad de envasado por línea y de mezcla en litros | `r3_capacidad` |
| 4 | Stock de seguridad $SS_{it}$ | `r4_stock_seguridad` |
| 5 | Horas extra acotadas | `r5_horas_extra` |
| 6 | Lote mínimo y *lead time* de jarabe | `r6_lote_minimo` (+ bloque 3) |
| 7 | Restricción de fin de horizonte | `r7_terminal` |
| 8 | Pedidos en firme de la semana 12 | `r8_pedidos_firme` |

**Entradas.** Los datos crudos de `datos/` y la tabla de traspaso de la Entrega 1
(`tabla_traspaso_entrega2.csv`), que se usa **tal como fue entregada**. Ningún archivo de entrada se
modifica: toda transformación ocurre en código.

**Cómo ejecutar.** Desde la raíz del repositorio: `python run_entrega2.py`. También puede abrirse en
Jupyter o Colab y ejecutarse de arriba hacia abajo.

In [1]:
# Preparación del entorno: ubica el paquete `entrega2` (carpeta "Entrega 2") y lo importa.
import sys
from pathlib import Path

carpeta = Path.cwd()
while not (carpeta / "entrega2").is_dir() and carpeta != carpeta.parent:
    carpeta = carpeta.parent
sys.path.insert(0, str(carpeta))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp as pl

from entrega2 import parametros as pa, modelo as mo, analisis as an, heuristicas as he

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pa.RESULTADOS.mkdir(parents=True, exist_ok=True)
pa.FIGURAS.mkdir(parents=True, exist_ok=True)
print("Paquete cargado desde:", carpeta)
print("PuLP", pl.__version__, "| solvers disponibles:", pl.listSolvers(onlyAvailable=True))

Paquete cargado desde: /home/claude/gdbs2/Entrega 2
PuLP 2.9.0 | solvers disponibles: ['PULP_CBC_CMD']


## 0. Revisión de los datos de entrada

El enunciado advierte que los datos provienen de un ERP real y **no están limpios**. Antes de
formular se revisan los archivos que usa esta entrega. Cada hallazgo queda con la decisión tomada.

In [2]:
tablas = pa.cargar_datos()
for nombre, df in tablas.items():
    print(f"{nombre:<14} {df.shape[0]:>5} filas x {df.shape[1]:>2} columnas")

hallazgos = pa.validar_datos(tablas)
display(hallazgos)

maestro            9 filas x 13 columnas
capacidad         13 filas x 14 columnas
inventario         9 filas x  5 columnas
setups             9 filas x  5 columnas
compromisos       15 filas x 10 columnas
demanda_hist    1404 filas x 10 columnas
traspaso         117 filas x 14 columnas


,Archivo,Chequeo,Resultado,Decisión
0,tabla_traspaso,"117 filas, 9 SKU x 13 semanas, sin duplicados","117 filas, duplicados=0",Se usa sin modificar.
1,tabla_traspaso,Valores faltantes en parámetros clave,0,Ninguno.
2,tabla_traspaso,Semanas con pronóstico 0,FUN-B30: 9,FUN-B30 es intermitente (69 de 156 semanas con venta 0); se respeta el pronóstico de la Entrega 1 y el SS cubre la incertidumbre.
3,tabla_traspaso,σ̂ desproporcionado respecto del pronóstico,"CIT-L05 h=4: σ̂=10,663 vs d̂=1,740; CIT-L05 h=9: σ̂=10,588 vs d̂=2,326","Origen: venta de 32.916 cajas de CIT-L05 el 29/12/2025 (vecinas ~2.600, sin promoción ni quiebre). No se corrige la Entrega 1; se evalúa en sensibilidad."
4,capacidad_horizonte,Horas regulares ya netas de mantención,L2 sem. 4 y 9: 40 h = 48 - 8; mezcla sem. 6: 49 h = 55 - 6,"Se usan las horas regulares tal cual, sin volver a descontar mantenciones."
5,capacidad_horizonte,Semana 13 con menos horas sin mantención declarada,"mezcla 47 h, L1 36 h, L2 40 h",Semana de Navidad (25/12). Se respeta el dato.
6,capacidad_horizonte,Tarifa de hora extra de L1,No existe columna para L1,Supuesto: misma tarifa que L2 (168.000 CLP/h).
7,inventario_inicial,"Inventario inicial, lote mínimo y nivel de servicio","9 SKU, sin negativos=True",Se usan tal cual. No hay inventario de jarabe ni atrasos iniciales.
8,compromisos_semana_focal,Cartera en firme mayor que el pronóstico de la semana 12,FUN-B30: 66 > 0,"La demanda de la semana 12 se toma como máx(pronóstico, cartera)."
9,compromisos_semana_focal,Línea y familia coherentes con el maestro,discrepancias=0,Ninguna.


**Lectura de la revisión.**

- La tabla de traspaso está completa (117 filas, sin faltantes). Se usa sin cambios, porque es el
  compromiso de la Entrega 1.
- La anomalía más importante es la **desviación del error de CIT-L05 en los horizontes 4 y 9**: es
  unas 6 veces el pronóstico. Viene de una venta registrada de 32.916 cajas el 29/12/2025. Las semanas
  vecinas rondan las 2.600 cajas, no hubo promoción ni quiebre de stock, y los otros SKU de la familia
  vendieron normal esa semana. Todo apunta a un error de digitación. No corregimos la Entrega 1:
  el escenario base usa la tabla tal cual y el impacto del dato se mide como sensibilidad en la Parte B.
- Las horas regulares de `capacidad_horizonte.csv` **ya vienen netas de mantención** (L2 trabaja 40 h
  en las semanas 4 y 9 = 48 − 8). Descontarlas de nuevo sería contar dos veces la misma pérdida.
- No existe tarifa de hora extra para L1. Se supone la misma de L2 y se declara.

In [3]:
# Contexto del dato atípico de CIT-L05 (solo lectura del histórico).
hist = tablas["demanda_hist"]
ventana = hist[(hist.fecha_lunes >= "2025-12-08") & (hist.fecha_lunes <= "2026-01-19")]
display(ventana.pivot(index="fecha_lunes", columns="sku", values="ventas_cajas")[["CIT-L05", "CIT-L15", "CIT-L30"]])

sku,CIT-L05,CIT-L15,CIT-L30
fecha_lunes,,,
2025-12-08,"2,609.00","2,924.00",673.00
2025-12-15,"2,435.00","2,820.00",544.00
2025-12-22,"2,555.00","3,615.00",471.00
2025-12-29,"32,916.00","2,320.00",628.00
2026-01-05,"2,584.00","3,013.00",512.00
2026-01-12,"2,866.00","2,667.00",562.00
2026-01-19,"2,661.00","2,267.00",715.00


## Conjuntos, índices y parámetros

| Símbolo | Definición | Fuente |
| :--- | :--- | :--- |
| $i \in \mathcal{I}$ | SKU (9 productos) | `maestro_sku.csv` |
| $t \in \{1,\dots,13\}$ | Semana del horizonte | `capacidad_horizonte.csv` |
| $\ell \in \{L1, L2\}$ | Línea de envasado; $\ell(i)$ es la línea del SKU $i$ | maestro |
| $f \in \{CIT, FUN, NEC\}$ | Familia; $f(i)$ es la familia del SKU $i$ | maestro |
| $\hat d_{it}$ | Pronóstico de demanda (cajas) | traspaso E1 |
| $\hat\sigma_{i,h}$ | Desviación estándar del error de pronóstico a horizonte $h=t$ | traspaso E1 |
| $m_{\ell}$ | Tasa de merma efectiva de la línea | traspaso E1 |
| $L^{ef}_i$ | Litros de jarabe por caja, con sobrellenado | traspaso E1 |
| $v_i$ | Tasa de envasado (cajas/h) | maestro |
| $H_{\ell t}, \bar E_{\ell t}$ | Horas regulares y máximo de horas extra de la línea | capacidad |
| $H^m_t, \bar E^m_t, r^m_t$ | Horas regulares, extra máximas y tasa (L/h) de la mezcladora | capacidad |
| $c_i, h_i, \pi_i$ | Costo variable, costo de inventario y penalidad de atraso | maestro |
| $I_{i0}, LM_i, \alpha_i$ | Inventario inicial, lote mínimo y nivel de servicio | inventario inicial |
| $C_i$ | Cartera en firme de la semana 12 | compromisos |

**Variables de decisión**

| Variable | Tipo | Significado |
| :--- | :--- | :--- |
| $x_{it}$ | continua $\ge 0$ | Cajas producidas (brutas, antes de merma) |
| $I_{it}$, $B_{it}$ | continuas $\ge 0$ | Inventario y atraso al cierre de la semana |
| $u_{it}$ | continua $\ge 0$ | Déficit respecto del stock de seguridad |
| $y_{it}$ | binaria | 1 si el SKU se prepara (produce) en la semana |
| $w_{\ell f t}$ | binaria | 1 si la línea $\ell$ abre una campaña de la familia $f$ en la semana |
| $n_{ft}$ | continua $\ge 0$ | Cambios de formato dentro de la familia $f$ en L2 |
| $E_{\ell t}$, $E^m_s$ | continuas $\ge 0$ | Horas extra de envasado y de mezcla |
| $V_{fs}$ | continua $\ge 0$ | Litros de jarabe de la familia $f$ mezclados en la semana $s$ |
| $z_{fs}$ | binaria | 1 si la mezcladora prepara jarabe de la familia $f$ en la semana $s$ |

In [4]:
P = pa.construir_parametros(pa.Escenario())
print("SKU:", P["SKUS"])
print("Familias:", P["FAMILIAS"], "| Líneas:", P["LINEAS"], "| Semanas:", P["T"][0], "a", P["T"][-1])
display(tablas["maestro"].set_index("sku")[["familia", "formato_L", "linea_envasado", "tasa_envasado_cajas_h",
                                            "costo_variable_caja_clp", "costo_inventario_caja_semana_clp",
                                            "penalidad_atraso_caja_semana_clp"]])

SKU: ['NEC-D05', 'NEC-D15', 'NEC-D30', 'CIT-L05', 'CIT-L15', 'CIT-L30', 'FUN-B05', 'FUN-B15', 'FUN-B30']
Familias: ['CIT', 'FUN', 'NEC'] | Líneas: ['L1', 'L2'] | Semanas: 1 a 13


,familia,formato_L,linea_envasado,tasa_envasado_cajas_h,costo_variable_caja_clp,costo_inventario_caja_semana_clp,penalidad_atraso_caja_semana_clp
sku,,,,,,,
NEC-D05,NEC,0.50,L1,180,3840,95,1400
NEC-D15,NEC,1.50,L2,210,5760,95,1400
NEC-D30,NEC,3.00,L2,130,7680,95,1400
CIT-L05,CIT,0.50,L1,180,2820,95,1400
CIT-L15,CIT,1.50,L2,210,4230,95,1400
CIT-L30,CIT,3.00,L2,130,5640,95,1400
FUN-B05,FUN,0.50,L1,180,4860,95,1400
FUN-B15,FUN,1.50,L2,210,7290,95,1400
FUN-B30,FUN,3.00,L2,130,9720,95,1400


# Punto 1 (1/8)
### 1. Balance de inventario por SKU y semana

Cada semana, lo disponible debe explicar lo atendido y lo que se difiere. Para el SKU $i$ en la
semana $t$, el **inventario neto** ($I_{it}-B_{it}$) cambia por la producción útil y la demanda:

$$
I_{it}-B_{it} \;=\; I_{i,t-1}-B_{i,t-1} \;+\; \rho_{\ell(i)}\,x_{it} \;-\; d_{it}
\qquad \forall i,t
$$

con $I_{it}\ge 0$ y $B_{it}\ge 0$. Para $t=1$ se usan el inventario inicial $I_{i0}$ del archivo y
$B_{i0}=0$, porque no hay atrasos declarados al inicio.

#### 1.1. Rendimiento heredado de la Entrega 1

La carta $p$ de la Entrega 1 entregó una tasa de merma $m_\ell$ por línea. El rendimiento es
$\rho_\ell = 1-m_\ell$, y $x_{it}$ representa la producción **bruta**. Por eso, para obtener $q$ cajas
vendibles hay que producir $q/(1-m_\ell)$, como pide el enunciado.

Por ejemplo, con $m_{L2}=3{,}11\,\%$, producir 1.000 cajas brutas deja 968,9 cajas vendibles.

La merma se aplica **una sola vez**, en la producción. No se descuenta de nuevo del inventario ni se
suma a la demanda.

#### 1.2. Pedidos atrasados (*backlogging*)

La demanda no atendida queda pendiente en $B_{it}$ y se arrastra a la semana siguiente. Cada caja
atrasada cuesta $\pi_i = 1.400$ CLP por **cada** semana que permanece pendiente:

$$
C_{\text{atraso}} = \sum_i\sum_t \pi_i\,B_{it}
$$

#### 1.3. Demanda de la semana 12

En la semana 12 hay cartera en firme. Si la cartera supera al pronóstico, el pronóstico subestima lo
que ya está vendido. Por eso se usa $d_{i,12}=\max(\hat d_{i,12},\,C_i)$, y en las demás semanas
$d_{it}=\hat d_{it}$.

In [5]:
rend = pd.DataFrame({"Línea": P["linea"], "Merma m (%)": {i: 100 * P["merma"][i] for i in P["SKUS"]},
                     "Rendimiento ρ (%)": {i: 100 * P["rend"][i] for i in P["SKUS"]},
                     "Cajas brutas por 1.000 vendibles": {i: 1000 / P["rend"][i] for i in P["SKUS"]}})
display(rend)

m_demo, v_demo = mo.construir_modelo(P, bloques=[mo.r1_balance])
print("Restricciones de balance:", len(m_demo.constraints), "(9 SKU x 13 semanas)")
print("Ejemplo, CIT-L15 semana 1:\n ", m_demo.constraints["balance_CIT_L15_1"])

,Línea,Merma m (%),Rendimiento ρ (%),Cajas brutas por 1.000 vendibles
NEC-D05,L1,1.92,98.08,"1,019.59"
NEC-D15,L2,3.11,96.89,"1,032.15"
NEC-D30,L2,3.11,96.89,"1,032.15"
CIT-L05,L1,1.92,98.08,"1,019.59"
CIT-L15,L2,3.11,96.89,"1,032.15"
CIT-L30,L2,3.11,96.89,"1,032.15"
FUN-B05,L1,1.92,98.08,"1,019.59"
FUN-B15,L2,3.11,96.89,"1,032.15"
FUN-B30,L2,3.11,96.89,"1,032.15"


Restricciones de balance: 117 (9 SKU x 13 semanas)
Ejemplo, CIT-L15 semana 1:
  -B_('CIT_L15',_1) + I_('CIT_L15',_1) - 0.9688542488020865*x_('CIT_L15',_1) = -563.27269338891


# Punto 2 (2/8)
### 2. Variables binarias de preparación y su enlace con la producción

Producir un SKU obliga a preparar la línea: limpiar, cambiar formato y ajustar el envasado. Eso
consume tiempo y dinero aunque se produzca una sola caja. Se modela con $y_{it}\in\{0,1\}$:

$$
x_{it} \le M_{it}\,y_{it} \qquad \forall i,t
$$

Si $y_{it}=0$, la producción es cero. Si $y_{it}=1$, puede producirse hasta $M_{it}$.

#### 2.1. Cota $M_{it}$ justificada

Un $M$ demasiado grande hace que la relajación lineal "pague" una fracción mínima de la preparación
($y_{it}=x_{it}/M$), y el solver tarda más en cerrar la brecha. Por eso se usa el **menor** de tres
límites, todos válidos porque ninguna solución óptima puede superarlos:

$$
M_{it}=\min\Big\{\underbrace{v_i\,(H_{\ell(i)t}+\bar E_{\ell(i)t})}_{\text{(a) línea completa}},\;
\underbrace{\tfrac{r^m_{t-1}\,(H^m_{t-1}+\bar E^m_{t-1}-\text{CIP})}{L^{ef}_i}}_{\text{(b) mezcla de } t-1},\;
\underbrace{\tfrac{\sum_{k\ge t} d_{ik}+\max_k SS_{ik}+T_i}{\rho_{\ell(i)}}}_{\text{(c) necesidad restante}}\Big\}
$$

- **(a)** Ni dedicando toda la línea (regular y extra) a un solo SKU se supera $v_i(H+\bar E)$.
- **(b)** El jarabe para envasar en $t$ se mezcla en $t-1$ (punto 6), así que tampoco puede superarse
  lo que la mezcladora alcanza a preparar esa semana.
- **(c)** Producir más que toda la demanda restante, más la mayor reserva y la reserva terminal, no
  tiene sentido económico.

#### 2.2. Campañas por familia

Las limpiezas CIP dependen de la **familia** y no del SKU (sección 2.3 del enunciado). Por eso se
agrega $w_{\ell f t}$: la línea abre una campaña de la familia $f$ si produce al menos uno de sus SKU.

$$
y_{it} \le w_{\ell(i) f(i) t}, \qquad w_{\ell f t} \le \sum_{i\in(\ell,f)} y_{it}
$$

In [6]:
filas = []
for i in P["SKUS"]:
    l = P["linea"][i]
    for t in (1, 6, 12):
        a = P["tasa"][i] * (P["H"][(l, t)] + P["Emax"][(l, t)])
        s = t - 1
        b = (P["Hm"][s] + P["Emax_m"][s] - P["cip_mezcla"][s]) * P["tasa_mezcla"][s] / P["litros_ef"][i]
        c = (sum(P["d"][(i, k)] for k in P["T"] if k >= t) + max(P["SS"][(i, k)] for k in P["T"])
             + P["T_terminal"][i]) / P["rend"][i]
        cotas = {"(a) línea": a, "(b) mezcla": b, "(c) necesidad": c}
        filas.append({"SKU": i, "Semana": t, **cotas, "M_it": P["M"][(i, t)],
                      "Cota activa": min(cotas, key=cotas.get)})
tabla_M = pd.DataFrame(filas)
display(tabla_M.round(0))
print("Cota activa (conteo sobre las 117 combinaciones):")
conteo = {}
for i in P["SKUS"]:
    for t in P["T"]:
        l, s = P["linea"][i], t - 1
        a = P["tasa"][i] * (P["H"][(l, t)] + P["Emax"][(l, t)])
        b = (P["Hm"][s] + P["Emax_m"][s] - P["cip_mezcla"][s]) * P["tasa_mezcla"][s] / P["litros_ef"][i]
        cc = (sum(P["d"][(i, k)] for k in P["T"] if k >= t) + max(P["SS"][(i, k)] for k in P["T"])
              + P["T_terminal"][i]) / P["rend"][i]
        k = min({"línea": a, "mezcla": b, "necesidad": cc}.items(), key=lambda z: z[1])[0]
        conteo[k] = conteo.get(k, 0) + 1
print(conteo)

,SKU,Semana,(a) línea,(b) mezcla,(c) necesidad,M_it,Cota activa
0,NEC-D05,1,"8,640.00","20,390.00","18,822.00","8,640.00",(a) línea
1,NEC-D05,6,"8,640.00","24,180.00","12,764.00","8,640.00",(a) línea
2,NEC-D05,12,"8,640.00","24,180.00","5,533.00","5,534.00",(c) necesidad
3,NEC-D15,1,"13,440.00","13,707.00","32,864.00","13,440.00",(a) línea
4,NEC-D15,6,"13,440.00","16,254.00","22,549.00","13,440.00",(a) línea
5,NEC-D15,12,"13,440.00","16,254.00","9,845.00","9,846.00",(c) necesidad
6,NEC-D30,1,"8,320.00","10,283.00","7,651.00","7,651.00",(c) necesidad
7,NEC-D30,6,"8,320.00","12,194.00","4,978.00","4,979.00",(c) necesidad
8,NEC-D30,12,"8,320.00","12,194.00","2,068.00","2,068.00",(c) necesidad
9,CIT-L05,1,"8,640.00","20,390.00","51,129.00","8,640.00",(a) línea


Cota activa (conteo sobre las 117 combinaciones):
{'línea': 64, 'necesidad': 53}


**Interpretación.** En L1 y en el formato de 1,5 L domina la cota de **línea**. En el formato de
3,0 L y hacia el final del horizonte, cuando queda poca demanda por cubrir, domina la de
**necesidad**. La cota de mezcla nunca es la menor: una semana de mezcladora alcanza para más de lo
que una línea puede envasar de un solo SKU. En ningún caso se usa un número arbitrario.

# Punto 3 (3/8)
### 3. Capacidad de envasado y de mezcla

#### 3.1. Tiempo de preparación en envasado

La matriz CIP es **asimétrica**. Pasar de cítrico a berries toma 1,0 h, pero volver toma 2,4 h. El
modelo táctico decide *qué* familias se envasan cada semana, pero no *en qué orden*. Por eso cada
campaña reserva el **tiempo esperado de entrada** a su familia, que es el promedio de la columna de
la matriz:

$$
\tau_f = \frac{1}{|\mathcal F|-1}\sum_{g\ne f} \text{CIP}_{g\to f}
$$

Este promedio tiene una propiedad útil. Con las tres familias activas hay dos ciclos posibles:
CIT→FUN→NEC→CIT, que suma 5,2 h, y CIT→NEC→FUN→CIT, que suma 6,2 h. Su promedio, 5,7 h, es
exactamente $\sum_f\tau_f$. El modelo supone una secuencia "promedio". Cuánto se equivoca con la
secuencia real es justamente lo que la Entrega Final debe medir.

En **L2**, que envasa 1,5 L y 3,0 L, cambiar de formato **dentro** de una familia agrega 0,6 h y
38.000 CLP. Si una campaña incluye los dos formatos, hay un cambio; en general,
$n_{ft}\ge \sum_{i\in(L2,f)}y_{it}-w_{L2,f,t}$. En **L1** cada familia tiene un solo SKU, así que el
cambio de formato de 0,4 h nunca ocurre.

La capacidad de cada línea queda:

$$
\sum_{i\in\ell}\frac{x_{it}}{v_i}\;+\;\sum_f \tau_f\,w_{\ell f t}\;+\;0{,}6\,\sum_f n_{ft}\,\mathbb 1[\ell=L2]
\;\le\; H_{\ell t}+E_{\ell t}\qquad\forall \ell,t
$$

In [7]:
setups = tablas["setups"].pivot(index="familia_origen", columns="familia_destino", values="horas_limpieza_cip")
print("Matriz CIP (horas), fila = sale, columna = entra:")
display(setups)
print("Tiempo esperado de entrada τ_f (h):", {f: round(v, 2) for f, v in P["tau_cip"].items()})
import itertools
for orden in itertools.permutations(["CIT", "FUN", "NEC"]):
    if orden[0] != "CIT":
        continue
    ciclo = list(orden) + [orden[0]]
    h = sum(setups.loc[a, b] for a, b in zip(ciclo, ciclo[1:]))
    print(" ciclo", " → ".join(ciclo), f"= {h:.1f} h")
print(f"Suma de τ_f = {sum(P['tau_cip'].values()):.1f} h (promedio de los ciclos)")

Matriz CIP (horas), fila = sale, columna = entra:


familia_destino,CIT,FUN,NEC
familia_origen,,,
CIT,0.00,1.00,1.00
FUN,2.40,0.00,1.40
NEC,2.80,2.80,0.00


Tiempo esperado de entrada τ_f (h): {'CIT': 2.6, 'FUN': 1.9, 'NEC': 1.2}
 ciclo CIT → FUN → NEC → CIT = 5.2 h
 ciclo CIT → NEC → FUN → CIT = 6.2 h
Suma de τ_f = 5.7 h (promedio de los ciclos)


#### 3.2. Capacidad de mezcla en litros

Una sola mezcladora prepara el jarabe de todas las familias a $r^m=2.300$ L/h. Cada familia que se
mezcla en la semana exige un CIP de 1,2 h. El jarabe por caja incluye el **sobrellenado** medido en la
Entrega 1: $L^{ef}_i = L^{nominal}_i + L^{sobrellenado}_i$. Las cajas que se pierden por merma
también consumen jarabe, porque se llenaron antes de descartarse.

$$
\frac{1}{r^m_s}\sum_f V_{fs} \;+\; 1{,}2\sum_f z_{fs} \;\le\; H^m_s + E^m_s,\qquad
V_{fs} \le r^m_s\,(H^m_s+\bar E^m_s)\,z_{fs}
$$

El jarabe mezclado en $s$ se envasa en $s+1$. Ese enlace se deduce en el punto 6.

In [8]:
lit = pd.DataFrame({"Litros nominales": tablas["maestro"].set_index("sku").litros_por_caja,
                    "Litros efectivos": P["litros_ef"]})
lit["Sobrellenado (L/caja)"] = lit["Litros efectivos"] - lit["Litros nominales"]
display(lit)

# ¿Cuántas horas de mezcla exige el pronóstico? (sin CIP, sin merma)
horas_req = {t: sum(P["litros_ef"][i] * P["d"][(i, t)] / P["rend"][i] for i in P["SKUS"]) / P["tasa_mezcla"][t]
             for t in P["T"]}
carga = pd.DataFrame({"Horas para el pronóstico (+merma)": horas_req,
                      "Horas + 3 CIP": {t: horas_req[t] + 3 * 1.2 for t in P["T"]},
                      "Horas regulares (semana anterior)": {t: P["Hm"][t - 1] for t in P["T"]},
                      "Regular + extra (semana anterior)": {t: P["Hm"][t - 1] + P["Emax_m"][t - 1] for t in P["T"]}})
carga.index.name = "Semana de envasado"
display(carga.round(1))
print("Semanas cuya demanda (con 3 CIP) supera la mezcla regular disponible:",
      [t for t in P["T"] if carga.loc[t, "Horas + 3 CIP"] > carga.loc[t, "Horas regulares (semana anterior)"]])

,Litros nominales,Litros efectivos,Sobrellenado (L/caja)
CIT-L05,6.00,6.07,0.07
CIT-L15,9.00,9.03,0.03
CIT-L30,12.00,12.03,0.03
FUN-B05,6.00,6.07,0.07
FUN-B15,9.00,9.03,0.03
FUN-B30,12.00,12.03,0.03
NEC-D05,6.00,6.07,0.07
NEC-D15,9.00,9.03,0.03
NEC-D30,12.00,12.03,0.03


,Horas para el pronóstico (+merma),Horas + 3 CIP,Horas regulares (semana anterior),Regular + extra (semana anterior)
Semana de envasado,,,,
1,38.90,42.50,55.00,55.00
2,43.90,47.50,55.00,65.00
3,44.20,47.80,55.00,65.00
4,48.30,51.90,55.00,65.00
5,44.90,48.50,55.00,65.00
6,46.70,50.30,55.00,65.00
7,48.40,52.00,49.00,59.00
8,49.70,53.30,55.00,65.00
9,50.40,54.00,55.00,65.00


Semanas cuya demanda (con 3 CIP) supera la mezcla regular disponible: [7, 11, 12]


**Lectura.** La carga de mezcla es alta en todo el horizonte y crece hacia el final. Solo para cubrir
el pronóstico y la merma, más tres CIP, se necesitan entre 42 y 59 horas por semana. En las
semanas de envasado 7, 11 y 12 eso supera las horas regulares de la semana anterior: la semana 6
tiene 6 h de mantención, y las semanas 11 y 12 son las previas al peak. A esa carga todavía hay que
sumar el stock de seguridad y la reserva terminal. Esto confirma la
pregunta de decisión de la Parte B: la mezcladora es el cuello de botella y habrá que elegir entre
horas extra, adelantar producción o atrasar.

# Punto 4 (4/8)
### 4. Stock de seguridad

#### 4.1. Deducción

La demanda real es el pronóstico más un error: $D_{it}=\hat d_{it}+\varepsilon_{it}$. Se supone que el
error del horizonte $h$ es aproximadamente normal, con media cero y desviación $\hat\sigma_{i,h}$, que
es la medida de la Entrega 1 por origen deslizante. Queremos que, con probabilidad $\alpha_i$, la
demanda de la semana no supere lo planificado más la reserva:

$$
P\big(D_{it}\le \hat d_{it}+SS_{it}\big)\ge\alpha_i
\;\Longleftrightarrow\;
P\!\left(\frac{\varepsilon_{it}}{\hat\sigma_{i,h}}\le\frac{SS_{it}}{\hat\sigma_{i,h}}\right)\ge\alpha_i
\;\Longleftrightarrow\;
\frac{SS_{it}}{\hat\sigma_{i,h}}\ge \Phi^{-1}(\alpha_i)
$$

La menor reserva que cumple, redondeada a cajas enteras, es:

$$
SS_{it}=\left\lceil \Phi^{-1}(\alpha_i)\,\hat\sigma_{i,h}\right\rceil,\qquad h=t
$$

Con $\alpha=95\,\%$, $\Phi^{-1}=1{,}645$, y con $\alpha=92\,\%$, $\Phi^{-1}=1{,}405$. Se usa el $\hat\sigma$ del
horizonte correspondiente, porque el plan se fija hoy para las 13 semanas: el error de la semana 9 es
el error a 9 semanas.

#### 4.2. Cómo entra al modelo

La reserva se mide sobre el **inventario neto**, porque una caja que ya se debe no protege a nadie.
Si una semana no alcanza la meta, el déficit $u_{it}$ se penaliza en lugar de volver infactible el
plan:

$$
I_{it}-B_{it}+u_{it}\ \ge\ SS_{it}, \qquad C_{SS}=\sum_{i,t}\pi^{SS}_i\,u_{it}
$$

El enunciado no da un costo para el déficit de reserva. Se usa $\pi^{SS}_i=\pi_i=1.400$ CLP por
caja-semana: quedar bajo la reserva expone, en el peor caso, a un quiebre que se paga al costo de
atraso. Este valor se varía en la sensibilidad de "costo de quiebre".

In [9]:
ss = pd.DataFrame({t: {i: P["SS"][(i, t)] for i in P["SKUS"]} for t in P["T"]})
print("Nivel de servicio α:", P["nivel_servicio"])
print("Stock de seguridad SS_it (cajas):")
display(ss)
rel = ss / pd.DataFrame({t: {i: P["pronostico"][(i, t)] for i in P["SKUS"]} for t in P["T"]}).replace(0, np.nan)
print("SS / pronóstico, valores sobre 1,5:")
display(rel.stack()[rel.stack() > 1.5].rename("SS / d̂").to_frame())

Nivel de servicio α: {'NEC-D05': 0.95, 'NEC-D15': 0.95, 'NEC-D30': 0.95, 'CIT-L05': 0.95, 'CIT-L15': 0.95, 'CIT-L30': 0.95, 'FUN-B05': 0.92, 'FUN-B15': 0.92, 'FUN-B30': 0.92}
Stock de seguridad SS_it (cajas):


,1,2,3,4,5,6,7,8,9,10,11,12,13
NEC-D05,801,439,605,1182,247,842,438,624,1167,346,847,512,646
NEC-D15,1631,1702,815,897,1161,1686,1758,1053,797,1264,1710,1931,1112
NEC-D30,155,244,180,182,177,280,246,183,190,243,289,260,176
CIT-L05,713,430,1085,17540,1501,690,377,1062,17416,1519,581,384,1034
CIT-L15,1047,1486,806,626,955,1097,1431,935,732,917,679,921,1048
CIT-L30,163,125,188,171,141,177,129,182,140,132,172,89,185
FUN-B05,1056,1052,719,957,590,1011,694,708,1074,544,960,655,1024
FUN-B15,682,270,750,1007,513,659,446,711,818,656,657,498,731
FUN-B30,210,232,132,297,220,293,267,132,282,294,300,284,153


SS / pronóstico, valores sobre 1,5:


SS / d̂
CIT-L05 4    10.08
        9     7.49

**Lectura.** Solo dos reservas superan 1,5 veces su pronóstico, y las dos son de **CIT-L05**, en
los horizontes 4 y 9: piden 17.540 y 17.416 cajas, entre 7 y 10 veces la demanda de esa semana y más
de dos semanas completas de L1. Es el efecto del dato atípico de la Entrega 1. **FUN-B30** tiene
pronóstico 0 en varias semanas por ser intermitente, así que su razón no está definida y no aparece
en la lista.

El escenario base mantiene la tabla tal cual, tal como se pidió. Como el déficit es penalizado y no
obligatorio, el modelo sigue siendo factible. La Parte B mide cuánto cuesta este dato.

# Punto 5 (5/8)
### 5. Horas extra como variable de decisión acotada

Las horas extra son una palanca de capacidad con costo explícito:

$$
0\le E_{\ell t}\le \bar E_{\ell t},\qquad 0\le E^m_s\le \bar E^m_s
$$

$$
C_{\text{extra}}=\sum_{\ell,t} c^{E}_{\ell}\,E_{\ell t}+\sum_s c^{E,m}\,E^m_s
$$

| Recurso | Máximo semanal | Costo (CLP/h) | Fuente |
| :--- | ---: | ---: | :--- |
| L1 | 8 h | 168.000 | **Supuesto**: los datos no traen tarifa para L1; se usa la de L2 |
| L2 | 16 h | 168.000 | enunciado y `capacidad_horizonte.csv` |
| Mezcla | 10 h | 121.000 | enunciado y `capacidad_horizonte.csv` |

In [10]:
cap = tablas["capacidad"].set_index("t")
display(cap[["horas_mezcla_regular", "horas_mezcla_extra_max", "horas_L1_regular", "horas_L1_extra_max",
             "horas_L2_regular", "horas_L2_extra_max", "mantencion_programada_h_L2",
             "mantencion_programada_h_mezcla"]])
print("Costos de hora extra usados:", P["c_extra"], "| mezcla:", P["c_extra_m"])

,horas_mezcla_regular,horas_mezcla_extra_max,horas_L1_regular,horas_L1_extra_max,horas_L2_regular,horas_L2_extra_max,mantencion_programada_h_L2,mantencion_programada_h_mezcla
t,,,,,,,,
1,55.00,10,40.00,8,48.00,16,0.00,0.00
2,55.00,10,40.00,8,48.00,16,0.00,0.00
3,55.00,10,40.00,8,48.00,16,0.00,0.00
4,55.00,10,40.00,8,40.00,16,8.00,0.00
5,55.00,10,40.00,8,48.00,16,0.00,0.00
6,49.00,10,40.00,8,48.00,16,0.00,6.00
7,55.00,10,40.00,8,48.00,16,0.00,0.00
8,55.00,10,40.00,8,48.00,16,0.00,0.00
9,55.00,10,40.00,8,40.00,16,8.00,0.00


Costos de hora extra usados: {'L1': 168000.0, 'L2': 168000.0} | mezcla: 121000.0


# Punto 6 (6/8)
### 6. Lote mínimo y *lead time* de jarabe

#### 6.1. Lote mínimo

Si se prepara la línea para un SKU, la corrida debe ser de al menos $LM_i$ cajas brutas:

$$
x_{it}\ \ge\ LM_i\,y_{it}
$$

Junto con $x_{it}\le M_{it}y_{it}$, la producción queda en $\{0\}\cup[LM_i, M_{it}]$.

#### 6.2. *Lead time* de jarabe de una semana

El jarabe preparado en la semana $s$ está disponible para envasar en $s+1$. Los litros que se envasan
en $t$ deben haberse mezclado en $t-1$:

$$
\sum_{i\in f} L^{ef}_i\,x_{it} \;=\; V_{f,t-1}\qquad \forall f,\ t=1,\dots,13
$$

**Supuestos declarados**

- **Sin inventario de jarabe.** El jarabe es perecible y no hay datos de estanques ni de su costo.
  Si se permitiera guardarlo gratis, el modelo adelantaría mezcla sin pagar nada y la mezcladora
  dejaría de ser una restricción real. Toda anticipación debe hacerse como **producto terminado**,
  que sí paga 95 CLP por caja-semana.
- **Semana 0.** El envasado de la semana 1 usa jarabe mezclado la semana anterior al horizonte
  (21/09). No hay registro de ese jarabe, así que se supone que se preparó con la capacidad
  **regular** de una semana normal (55 h, sin horas extra). Suponer cero impediría envasar en la
  semana 1 y crearía un atraso artificial.
- **Semana 13.** La mezcla de la semana 13 alimenta la semana 14, fuera del horizonte. No se modela.
  La reserva terminal del punto 7 cubre esa continuidad.

In [11]:
display(pd.DataFrame({"Lote mínimo (cajas)": P["lote_min"],
                      "Lead time jarabe (semanas)": tablas["inventario"].set_index("sku").lead_time_jarabe_semanas,
                      "Inventario inicial": P["I0"]}))
print("Semanas de mezcla modeladas:", P["SEMANAS_MEZCLA"], "(la mezcla de s alimenta el envasado de s+1)")

,Lote mínimo (cajas),Lead time jarabe (semanas),Inventario inicial
NEC-D05,350.00,1,"1,024.00"
NEC-D15,600.00,1,"3,162.00"
NEC-D30,140.00,1,339.00
CIT-L05,350.00,1,"1,217.00"
CIT-L15,430.00,1,"1,282.00"
CIT-L30,70.00,1,255.00
FUN-B05,550.00,1,"2,135.00"
FUN-B15,360.00,1,"1,261.00"
FUN-B30,30.00,1,155.00


Semanas de mezcla modeladas: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] (la mezcla de s alimenta el envasado de s+1)


# Punto 7 (7/8)
### 7. Restricción de fin de horizonte

#### 7.1. Por qué hace falta

El horizonte termina el 21/12/2026, justo antes del peak de verano. Si nada lo impide, minimizar
costos lleva a **vaciar el inventario** en las últimas semanas: lo que se deja en la semana 13 solo
genera costo de inventario y no aporta nada dentro del horizonte. Peor aún, atrasar una caja cuesta
1.400 CLP por semana y producirla cuesta entre 2.820 y 9.720 CLP. Sin una condición terminal, el
modelo prefiere **no producir** al final y dejar los pedidos pendientes para enero, donde "no
cuestan". La Parte B lo demuestra resolviendo el modelo sin esta restricción.

#### 7.2. Regla propuesta

Al cierre de la semana 13, el inventario neto debe cubrir la reserva de seguridad **más** una semana
de demanda esperada de enero:

$$
I_{i,13}-B_{i,13}\ \ge\ T_i,\qquad
T_i=\left\lceil SS_{i,13}+\tilde d_{i,14}\right\rceil,\qquad
\tilde d_{i,14}=\Big(\tfrac14\sum_{t=10}^{13}\hat d_{it}\Big)\cdot\phi_{f(i)}
$$

- Se mide sobre inventario neto: no se puede "tener" reserva mientras se le deben cajas a clientes.
- $\phi_f$ es el **factor estacional post-horizonte** de la familia, estimado del histórico: la razón
  entre la mediana de ventas de las 4 semanas posteriores al 21-dic y la de las 4 semanas anteriores,
  para cada año disponible. Se usa la mediana entre años y SKU, para que un dato aislado (como el de
  CIT-L05) no la mueva.
- **No es un pronóstico de enero**, que la Entrega 1 no entregó. Es una regla de continuidad: la
  planta debe empezar el próximo plan con una semana de cobertura más su reserva.

In [12]:
print("Factor estacional post-horizonte φ_f:", {f: round(v, 3) for f, v in P["factor_estacional"].items()})
term = pd.DataFrame({"Familia": P["familia"],
                     "Promedio d̂ sem. 10-13": {i: np.mean([P["d"][(i, t)] for t in (10, 11, 12, 13)]) for i in P["SKUS"]},
                     "φ_f": {i: P["factor_estacional"][P["familia"][i]] for i in P["SKUS"]},
                     "d̃ semana 14": P["d14"], "SS_i,13": {i: P["SS"][(i, 13)] for i in P["SKUS"]},
                     "Reserva terminal T_i": P["T_terminal"]})
display(term.round(1))

Factor estacional post-horizonte φ_f: {'CIT': 0.989, 'FUN': 1.099, 'NEC': 1.094}


,Familia,Promedio d̂ sem. 10-13,φ_f,d̃ semana 14,"SS_i,13",Reserva terminal T_i
NEC-D05,NEC,"1,208.00",1.10,"1,322.00",646,1968
NEC-D15,NEC,"2,068.80",1.10,"2,264.00",1112,3376
NEC-D30,NEC,485.90,1.10,531.80,176,708
CIT-L05,CIT,"2,544.50",1.00,"2,516.10",1034,3551
CIT-L15,CIT,"3,009.90",1.00,"2,976.40",1048,4025
CIT-L30,CIT,580.20,1.00,573.80,185,759
FUN-B05,FUN,"2,685.70",1.10,"2,951.70",1024,3976
FUN-B15,FUN,"1,808.50",1.10,"1,987.60",731,2719
FUN-B30,FUN,97.50,1.10,107.20,153,261


# Punto 8 (8/8)
### 8. Pedidos en firme de la semana 12

La semana 12 (14/12/2026) tiene una cartera de 15 órdenes con día y hora de entrega. Sumadas por SKU
dan $C_i$. El enunciado exige que **la producción de la semana 12** cubra, como mínimo, esa cartera.
Como $x$ es bruta, se exige en cajas vendibles:

$$
\rho_{\ell(i)}\,x_{i,12}\ \ge\ C_i\qquad\forall i
$$

Esto obliga a producir los 9 SKU en la semana 12. La programación detallada de la Entrega Final
necesita esos lotes para secuenciar las órdenes por fecha y hora, así que no basta con despacharlas
desde inventario.

Además, la demanda de la semana 12 usa $\max(\hat d_{i,12},C_i)$ (punto 1.3). En FUN-B30 el pronóstico
es 0, pero hay 66 cajas vendidas en firme.

In [13]:
com = tablas["compromisos"]
resumen = com.groupby("sku").agg(ordenes=("orden", "count"), cajas=("cantidad_comprometida_cajas", "sum"))
resumen["Pronóstico sem. 12"] = [P["pronostico"][(i, 12)] for i in resumen.index]
resumen["Demanda usada sem. 12"] = [P["d"][(i, 12)] for i in resumen.index]
resumen["Producción bruta mínima"] = [P["C"][i] / P["rend"][i] for i in resumen.index]
resumen["Lote mínimo"] = [P["lote_min"][i] for i in resumen.index]
display(resumen.round(1))

,ordenes,cajas,Pronóstico sem. 12,Demanda usada sem. 12,Producción bruta mínima,Lote mínimo
sku,,,,,,
CIT-L05,2,839,"2,435.00","2,435.00",855.40,350.00
CIT-L15,2,1026,"3,433.30","3,433.30","1,059.00",430.00
CIT-L30,1,157,570.70,570.70,162.00,70.00
FUN-B05,2,1340,"2,861.10","2,861.10","1,366.30",550.00
FUN-B15,2,860,"1,559.20","1,559.20",887.60,360.00
FUN-B30,1,66,0.00,66.00,68.10,30.00
NEC-D05,2,898,"1,023.00","1,023.00",915.60,350.00
NEC-D15,2,1320,"2,231.40","2,231.40","1,362.40",600.00
NEC-D30,1,313,546.00,546.00,323.10,140.00


# 9. Función objetivo

Se minimiza el costo total del plan en las 13 semanas:

$$
\begin{aligned}
\min Z = {} & \sum_{i,t} c_i\,x_{it} \;+\; \sum_{i,t} h_i\,I_{it} \;+\; \sum_{i,t}\pi_i\,B_{it} \;+\; \sum_{i,t}\pi^{SS}_i\,u_{it}\\
&+\; 74.000\sum_{\ell,f,t}\tau_f\,w_{\ell f t} \;+\; 38.000\sum_{f,t} n_{ft} \;+\; 74.000\cdot 1{,}2\sum_{f,s} z_{fs}\\
&+\; \sum_{\ell,t} c^E_\ell\,E_{\ell t} \;+\; 121.000\sum_s E^m_s
\end{aligned}
$$

| Término | Criterio |
| :--- | :--- |
| Costo variable | Se aplica a la producción **bruta**: las cajas mermadas también consumieron insumos. |
| Inventario | 95 CLP por caja y semana sobre el inventario al cierre. |
| Atraso | 1.400 CLP por caja y semana pendiente. |
| Déficit de SS | Supuesto de política, $\pi^{SS}=\pi$ (punto 4.2). |
| CIP envasado | 74.000 CLP/h por el tiempo esperado de entrada a cada familia activa. |
| Cambio de formato | 38.000 CLP por cambio en L2. |
| CIP mezcla | 1,2 h por familia mezclada, valorizadas a la misma tarifa CIP (supuesto). |
| Horas extra | Según tarifa por recurso (punto 5). |

El costo variable pesa mucho en el total, pero **casi no cambia entre planes factibles**: la reserva
terminal fija el volumen total que hay que producir. Por eso la Parte B compara planes con el **costo
relevante**, es decir, el costo total sin el costo variable.

# 10. Modelo completo

$$
\begin{aligned}
\min\ & Z \quad\text{(sección 9)}\\
\text{s.a.}\ & I_{it}-B_{it}=I_{i,t-1}-B_{i,t-1}+\rho_{\ell(i)}x_{it}-d_{it} && \forall i,t &(1)\\
& x_{it}\le M_{it}\,y_{it},\quad y_{it}\le w_{\ell(i)f(i)t},\quad w_{\ell ft}\le\textstyle\sum_{i\in(\ell,f)}y_{it} && \forall i,t;\ \forall \ell,f,t &(2)\\
& \textstyle\sum_{i\in\ell}x_{it}/v_i+\sum_f\tau_f w_{\ell ft}+0{,}6\sum_f n_{ft}\mathbb 1[\ell=L2]\le H_{\ell t}+E_{\ell t} && \forall \ell,t &(3a)\\
& n_{ft}\ge\textstyle\sum_{i\in(L2,f)}y_{it}-w_{L2,f,t} && \forall f,t &(3b)\\
& \textstyle\sum_f V_{fs}/r^m_s+1{,}2\sum_f z_{fs}\le H^m_s+E^m_s,\quad V_{fs}\le r^m_s(H^m_s+\bar E^m_s)z_{fs} && s=0,\dots,12 &(3c)\\
& I_{it}-B_{it}+u_{it}\ge SS_{it} && \forall i,t &(4)\\
& E_{\ell t}\le\bar E_{\ell t},\quad E^m_s\le\bar E^m_s && &(5)\\
& x_{it}\ge LM_i\,y_{it},\qquad \textstyle\sum_{i\in f}L^{ef}_i x_{it}=V_{f,t-1} && \forall i,t;\ \forall f,t &(6)\\
& I_{i,13}-B_{i,13}\ge T_i && \forall i &(7)\\
& \rho_{\ell(i)}\,x_{i,12}\ge C_i && \forall i &(8)\\
& x,I,B,u,n,E,E^m,V\ge 0;\quad y,w,z\in\{0,1\}
\end{aligned}
$$

In [14]:
modelo, var = mo.construir_modelo(P)
grupos = {"(1) balance_": "balance_", "(2) enlace_": "enlace_", "(2) campana_": "campana_",
          "(3) cap_L": "cap_L", "(3) formato_": "formato_", "(3) cap_mezcla_": "cap_mezcla_",
          "(3) activa_mezcla_": "activa_mezcla_", "(4) ss_": "ss_", "(5) extra_max_": "extra_max_",
          "(6) lote_min_": "lote_min_", "(6) jarabe_": "jarabe_", "(7) terminal_": "terminal_",
          "(8) firme_": "firme_"}
estructura = pd.Series({g: sum(1 for n in modelo.constraints if n.startswith(p) and
                               not (p == "campana_" and n.startswith("campana_util")))
                        for g, p in grupos.items()}, name="Restricciones")
display(estructura.to_frame())
binarias = sum(1 for x in modelo.variables() if x.cat == pl.LpInteger)
print(f"Variables: {len(modelo.variables())} (binarias: {binarias}) | Restricciones: {len(modelo.constraints)}")

,Restricciones
(1) balance_,117
(2) enlace_,117
(2) campana_,117
(3) cap_L,26
(3) formato_,39
(3) cap_mezcla_,13
(3) activa_mezcla_,39
(4) ss_,117
(5) extra_max_,39
(6) lote_min_,117


Variables: 819 (binarias: 234) | Restricciones: 876


#### Verificación de factibilidad con la relajación lineal

Antes de pasar a la Parte B se comprueba que el modelo no tenga contradicciones. Se resuelve la
relajación lineal, con las binarias continuas en [0, 1]. Si la relajación es infactible, el MIP
también lo es. Si es factible, su valor es una cota inferior del costo óptimo.

In [15]:
relajado, var_r = mo.construir_modelo(P)
for nombre in ("y", "w", "z"):
    for x in var_r[nombre].values():
        x.cat = pl.LpContinuous
relajado.solve(pl.PULP_CBC_CMD(msg=False))
print("Estado de la relajación lineal:", pl.LpStatus[relajado.status])
print(f"Cota inferior del costo total: {pl.value(relajado.objective)/1e6:,.1f} MM CLP")
assert pl.LpStatus[relajado.status] == "Optimal"

Estado de la relajación lineal: Optimal
Cota inferior del costo total: 980.0 MM CLP


## Supuestos declarados de la formulación

1. Tabla de traspaso de la Entrega 1 usada **sin modificaciones** (incluido el $\hat\sigma$ de CIT-L05).
2. Demanda de la semana 12 igual a $\max(\hat d_{i,12}, C_i)$.
3. Stock de seguridad blando sobre inventario neto, con $\pi^{SS}=\pi=1.400$ CLP/caja-semana.
4. Preparaciones con tiempo esperado de entrada por familia ($\tau_f$) y una campaña por familia
   activa, línea y semana. La secuencia real se contrasta en la Entrega Final.
5. Tarifa de hora extra de L1 igual a la de L2 (168.000 CLP/h).
6. CIP de mezcla valorizado a 74.000 CLP/h.
7. Sin inventario de jarabe; mezcla de la semana 0 con capacidad regular.
8. Reserva terminal $T_i = SS_{i,13} + $ una semana de demanda esperada ajustada por estacionalidad.

La Parte B resuelve este modelo y analiza su sensibilidad a estos supuestos.